In [25]:
# 1. Install dependencies if needed:
# pip install roboflow torch torchvision pillow

from roboflow import Roboflow
import os
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# ====== Roboflow config ======
API_KEY = "EEBeol2sg2VYzNa1xz1S"
WORKSPACE = "tea-project-jboyg"
PROJECT_NAME = "camellia_idrone"
VERSION_NUMBER = 2

# ====== Download dataset ======
rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
version = project.version(VERSION_NUMBER)

print("Downloading dataset locally, please wait...")
dataset_obj = version.download("yolov11")  # downloads dataset locally
dataset_dir = dataset_obj.location  # local dataset folder path
print(f"Dataset downloaded to: {dataset_dir}")

# ====== Dataset class loading images and labels ======
class LocalYoloDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.image_files = sorted(list(self.images_dir.glob("*.jpg")))
        self.transform = transform or T.Compose([
            T.Resize((640, 640)),
            T.ToTensor(),
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.labels_dir / f"{img_path.stem}.txt"

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        boxes = []
        if label_path.exists():
            with open(label_path, "r") as f:
                for line in f:
                    cls, x, y, w, h = map(float, line.strip().split())
                    boxes.append([cls, x, y, w, h])
        targets = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0,5))

        return image, targets

# ====== Prepare paths and dataloader ======
# For example, if the images are inside a subfolder 'train' or similar
images_dir = os.path.join(dataset_dir, "train", "images")
labels_dir = os.path.join(dataset_dir, "train", "labels")


dataset = LocalYoloDataset(images_dir, labels_dir)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=lambda batch: tuple(zip(*batch)))

print(f"Dataset size: {len(dataset)} images")

# ====== Example batch iteration ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for images, targets in dataloader:
    images = [img.to(device) for img in images]
    targets = [t.to(device) for t in targets]
    print(f"Batch of {len(images)} images")
    print(f"Example targets shape: {targets[0].shape if len(targets) > 0 else 'No targets'}")
    break

# ====== Next step: You can plug this dataloader into your YOLOv11 training loop ======


loading Roboflow workspace...
loading Roboflow project...
Dataset downloaded to: /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [26]:
from roboflow import Roboflow

API_KEY = "EEBeol2sg2VYzNa1xz1S"
WORKSPACE = "tea-project-jboyg"
PROJECT_NAME = "camellia_idrone"
VERSION_NUMBER = 2

rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
version = project.version(VERSION_NUMBER)

print("Starting download...")
dataset_obj = version.download("yolov11")  # or "yolov5" if you want
print(f"Download finished! Dataset location: {dataset_obj.location}")


loading Roboflow workspace...
loading Roboflow project...
Starting download...
Download finished! Dataset location: /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2


In [ ]:
# Install required packages (uncomment if running in fresh environment)
# !pip install roboflow ultralytics

from roboflow import Roboflow
from ultralytics import YOLO
import os

# === Step 1: Download dataset from Roboflow ===
API_KEY = "EEBeol2sg2VYzNa1xz1S"
WORKSPACE = "tea-project-jboyg"
PROJECT = "camellia_idrone"
VERSION = 2

print("Downloading dataset from Roboflow...")
rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)
dataset = version.download("coco")
print(f"Dataset downloaded to: {dataset.location}")

# === Step 2: Set path to dataset config ===
data_yaml = os.path.join(dataset.location, "data.yaml")
print(f"Using dataset config file at: {data_yaml}")

# === Step 3: Initialize YOLOv11 model ===
model = YOLO("yolov11n.pt")  # yolov11 nano weights

# === Step 4: Train the model ===
print("Training YOLOv11 model...")
results = model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=8,
    name="camellia_yolov11_experiment",
    device=0  # adjust if you have multiple GPUs or CPU only
)

# === Step 5: Evaluate model with confusion matrix and metrics ===
print("Evaluating the trained model...")
metrics = model.val(
    data=data_yaml,
    conf=0.001,
    iou=0.65,
    plots=True
)
print(metrics)


loading Roboflow workspace...
loading Roboflow project...
Dataset downloaded to: /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2
Using dataset config file at: /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/data.yaml


FileNotFoundError: [Errno 2] No such file or directory: 'yolov11n.pt'

In [31]:
# !pip install roboflow ultralytics

import os
from roboflow import Roboflow
from ultralytics import YOLO

API_KEY = "EEBeol2sg2VYzNa1xz1S"
WORKSPACE = "tea-project-jboyg"
PROJECT = "camellia_idrone"
VERSION = 2

# Step 1: Download dataset from Roboflow in COCO format
rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)
dataset = version.download("coco")
print(f"Dataset downloaded to: {dataset.location}")

data_yaml = os.path.join(dataset.location, "data.yaml")
print(f"Using dataset config file at: {data_yaml}")

# Step 2: Load your existing model weights file (replace 'best_model.pt' with your actual filename)
model_weights_path = "/home/idrone2/Desktop/Ranjith-works/yolo/yolo11n.pt"  # <-- your existing weights file

if not os.path.exists(model_weights_path):
    raise FileNotFoundError(f"Model weights file not found: {model_weights_path}")

model = YOLO(model_weights_path)

# Step 3: Train or evaluate your model

# For training (optional if you want to continue training):
model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=8,
    name="camellia_idrone_training",
    device=0  # GPU index
)

# For validation and metrics (after training or just to evaluate):
metrics = model.val(data=data_yaml, conf=0.001, iou=0.65, plots=True)
print(metrics)


loading Roboflow workspace...
loading Roboflow project...
Dataset downloaded to: /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2
Using dataset config file at: /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/data.yaml
New https://pypi.org/project/ultralytics/8.3.167 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.65 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A2000 12GB, 11926MiB)
engine/trainer: task=detect, mode=train, model=/home/idrone2/Desktop/Ranjith-works/yolo/yolo11n.pt, data=/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/data.yaml, epochs=50, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=None, name=camellia_idrone_training, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_sca

RuntimeError: Dataset '/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/data.yaml' error ❌ '/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/data.yaml' does not exist

In [32]:
import os
import zipfile
from ultralytics import YOLO

# === Step 1: Set paths ===
zip_path = "/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/roboflow.zip"  # Change this to your downloaded ZIP file path
dataset_dir = "/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/dataset"        # Where to unzip dataset
model_weights = "home/idrone2/Desktop/Ranjith-works/yolo/yolov11n.pt"                    # Pretrained weights or your model path

# Create dataset directory if doesn't exist
os.makedirs(dataset_dir, exist_ok=True)

# === Step 2: Unzip dataset ===
print(f"Unzipping dataset from {zip_path} to {dataset_dir} ...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(dataset_dir)

# === Step 3: Verify data.yaml ===
data_yaml = os.path.join(dataset_dir, "data.yaml")
if not os.path.isfile(data_yaml):
    raise FileNotFoundError(f"data.yaml not found in {dataset_dir}! Check your unzip folder structure.")

print(f"Found dataset config: {data_yaml}")

# === Step 4: Load YOLOv11 model ===
print(f"Loading model weights: {model_weights}")
model = YOLO(model_weights)

# === Step 5: Train the model ===
print("Starting training...")
model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=8,
    name="roboflow_yolov11_training",
    device=0  # set to your GPU device number or 'cpu' if no GPU
)

# === Step 6: Evaluate & get metrics ===
print("Evaluating model on validation set...")
metrics = model.val(
    data=data_yaml,
    conf=0.001,   # confidence threshold
    iou=0.65,     # IoU threshold
    plots=True    # Show plots like confusion matrix
)

print("Evaluation metrics:", metrics.metrics)


Unzipping dataset from /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/roboflow.zip to /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/dataset ...


BadZipFile: File is not a zip file

In [33]:
import os

zip_path = "/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/roboflow.zip"
print(f"Exists? {os.path.exists(zip_path)}")
print(f"Size (bytes): {os.path.getsize(zip_path)}")
print(f"Extension: {os.path.splitext(zip_path)[1]}")


Exists? True
Size (bytes): 11330101248
Extension: .zip


In [1]:

from roboflow import Roboflow
from ultralytics import YOLO

In [2]:
# Step 1: Download and extract dataset from Roboflow (YOLOv11 format)
rf = Roboflow(api_key="EEBeol2sg2VYzNa1xz1S")
project = rf.workspace("tea-project-jboyg").project("camellia_idrone")
version = project.version(2)
dataset = version.download("yolov11")  # make sure you ask for 'yolov11' format

print(f"Dataset downloaded and extracted at: {dataset.location}")

loading Roboflow workspace...
loading Roboflow project...
Dataset downloaded and extracted at: /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2


In [8]:
# Step 2: Load your YOLOv11 model (change to your model path or pretrained weight)
model = YOLO("/home/idrone2/Desktop/Ranjith-works/yolo11n.pt")  # or path to your own weights


In [7]:
import os
print(os.path.exists("/home/idrone2/Desktop/Ranjith-works/yolo11n.pt"))


True


In [9]:
# Step 3: Train the model on the downloaded dataset
model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    name="camellia_idrone_training",
    device=0  # change or remove if you want to use CPU
)

New https://pypi.org/project/ultralytics/8.3.167 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.65 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A2000 12GB, 11926MiB)
engine/trainer: task=detect, mode=train, model=/home/idrone2/Desktop/Ranjith-works/yolo11n.pt, data=/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/data.yaml, epochs=50, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=None, name=camellia_idrone_training2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=Fals

RuntimeError: Dataset '/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/data.yaml' error ❌ '/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/data.yaml' does not exist

In [1]:
from roboflow import Roboflow

rf = Roboflow(api_key="EEBeol2sg2VYzNa1xz1S")
project = rf.workspace("tea-project-jboyg").project("camellia_idrone")
version = project.version(2)

# Download in YOLO format (YOLOv5, YOLOv7, YOLOv8 formats should be compatible with YOLOv11)
dataset = version.download("yolov5")  
print("Dataset downloaded at:", dataset.location)


loading Roboflow workspace...
loading Roboflow project...
Dataset downloaded at: /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2


In [ ]:
import json
import os
from PIL import Image

# Define class name to ID map
class_map = {
    "RSM_Moderate": 0,
    "RSM_Severe": 1
}

def convert_json_to_yolo(json_folder, image_folder):
    for filename in os.listdir(json_folder):
        if filename.endswith(".json"):
            json_path = os.path.join(json_folder, filename)
            with open(json_path, 'r') as f:
                data = json.load(f)
            
            # assumes Roboflow JSON format (you might need to tweak this based on your exact format)
            image_filename = data['image']['filename']
            image_path = os.path.join(image_folder, image_filename)
            img = Image.open(image_path)
            img_w, img_h = img.size
            
            yolo_lines = []
            for ann in data['annotations']:
                class_name = ann['class']
                class_id = class_map[class_name]
                bbox = ann['bbox']
                x = bbox['x']
                y = bbox['y']
                w = bbox['width']
                h = bbox['height']
                
                # Convert to YOLO format
                x_center = (x + w / 2) / img_w
                y_center = (y + h / 2) / img_h
                w /= img_w
                h /= img_h

                yolo_lines.append(f"{class_id} {x_center} {y_center} {w} {h}")
            
            # Save YOLO format txt
            label_filename = os.path.splitext(image_filename)[0] + ".txt"
            label_path = os.path.join(json_folder, label_filename)
            with open(label_path, 'w') as f:
                f.write("\n".join(yolo_lines))

# Example usage
root_folder = "/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone"

for split in ["train", "val", "test"]:
    print(f"Converting {split}...")
    convert_folder(os.path.join(base_dir, split))


KeyError: 'image'